In [0]:
%sql
-- EDA inicial
-- conteo de registros 

select count(*) as total_registros 
from tiktok_data_eng.bronze.tiktok_bronze

In [0]:
%sql
-- muestra de datos para verificar datos
select *
from tiktok_data_eng.bronze.tiktok_bronze
limit(5);


In [0]:
%sql
-- lateral explode del campo hashtags
-- el campo hashtags se guarda como string con formato Python: ['tag1', 'tag2', ...]
-- se reemplazan las comillas simples por dobles para que sea JSON válido y luego se parsea como array<string>

create or replace temporary view registros_completos as

  select
  search_type,
  search_hashtag,
  video_id,
  description,
  created_at,
  create_time,
  video_url,
  region,
  duration,
  video_width,
  video_height,
  ratio,
  definition,
  bitrate,
  cover,
  is_ad,
  is_photo,
  is_paid_content,
  description_language,
  duet_control,
  stitch_control,
  pinned,
  plays,
  likes,
  comments,
  shares,
  saves,
  reposts,
  author_id,
  author_sec_uid,
  author_unique_id,
  author_nickname,
  author_verified,
  author_verify_reason,
  author_signature,
  author_verification_type,
  author_region,
  author_followers,
  author_following,
  author_hearts,
  author_videos,
  author_favoriting,
  author_instagram
  author_youtube,
  music_id,
  music_title,
  music_author,
  music_is_original,
  music_duration,
  hashtags_exploded,
  mentions,
  fecha_ingesta
  from tiktok_data_eng.bronze.tiktok_bronze
  lateral view explode(
    from_json(regexp_replace(hashtags, "'", '"'), 'array<string>')
  ) as hashtags_exploded
  where hashtags_exploded is not null
    and hashtags_exploded != 'None'

In [0]:
%sql
select * from registros_completos
limit 100;

In [0]:
%sql
-- conteo de valores diferentes de cero para stitch y duet 
select
sum(case when cast(duet_control as double) = 1  then 1 else 0 end) as conteo_valores_diferentes_duet,
sum(case when cast(stitch_control as double) = 1 then 1 else 0 end) as conteo_valores_diferentes_stitch
from registros_completos;


In [0]:
%sql
-- contar valores de author_verified, author_verification_type, author_favoriting, definition y create_time
-- con casteos apropiados: author_verified -> boolean, verification_type -> int, favoriting -> int, definition -> None a NULL, create_time -> int
SELECT
    COUNT(*) AS total_registros,

    -- Verificación del autor (cast a boolean)
    SUM(CASE WHEN CAST(author_verified AS BOOLEAN) = TRUE THEN 1 ELSE 0 END) AS verified_true,
    SUM(CASE WHEN CAST(author_verified AS BOOLEAN) = FALSE THEN 1 ELSE 0 END) AS verified_false,
    SUM(CASE WHEN CAST(author_verified AS BOOLEAN) IS NULL THEN 1 ELSE 0 END) AS verified_null,

    -- Tipo de verificación (cast a entero)
    SUM(CASE WHEN CAST(author_verification_type AS INT) IS NULL THEN 1 ELSE 0 END) AS verification_type_null,
    SUM(CASE WHEN CAST(author_verification_type AS INT) IS NOT NULL THEN 1 ELSE 0 END) AS verification_type_no_null,

    -- Favoriting (cast a entero)
    SUM(CASE WHEN CAST(author_favoriting AS INT) = 0 THEN 1 ELSE 0 END) AS favoriting_cero,
    SUM(CASE WHEN CAST(author_favoriting AS INT) IS NULL THEN 1 ELSE 0 END) AS favoriting_null,
    SUM(CASE WHEN CAST(author_favoriting AS INT) > 0 THEN 1 ELSE 0 END) AS favoriting_mayor_cero,

    -- Mentions
    SUM(CASE WHEN mentions = '[]' THEN 1 ELSE 0 END) AS mentions_vacios,
    SUM(CASE WHEN mentions IS NULL THEN 1 ELSE 0 END) AS mentions_null,

    -- Definition (transformar 'None' a NULL y contar nulos)
    SUM(CASE WHEN NULLIF(definition, 'None') IS NULL THEN 1 ELSE 0 END) AS definition_null_o_none,
    SUM(CASE WHEN definition = 'None' THEN 1 ELSE 0 END) AS definition_valor_none,

    -- Create time (cast a entero)
    SUM(CASE WHEN CAST(create_time AS BIGINT) IS NULL THEN 1 ELSE 0 END) AS create_time_null

FROM registros_completos;